# Character Model (CM) Training

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/enph-479-edge-ai-stt/enph-479-edge-ai-stt/blob/main/training/notebooks/cm_training.ipynb)

Trains the character-level language model (2x256 unidirectional LSTM) on a random sample of the LibriSpeech LM text and prints dev-clean bits per character after every epoch. **Runtime: T4 GPU.** Runs start to finish in one session, then saves the weights and the training log to a run folder on Google Drive.

Follows the paper's recipe: randomly selected sentences concatenated into one stream with EOS between them, one-hot character in, next-character distribution out, truncated BPTT with the LSTM state carried across chunks. The LM shares the AM's 30-symbol vocab (provisional until `shared/specs` is frozen); the CTC blank never occurs in text. One departure: Adam instead of the paper's AdaDelta, like the AM, since AdaDelta converged much more slowly.

## 1. Pull the repo

In [ ]:
import importlib
import os
import subprocess
import sys

REPO_URL = "https://github.com/enph-479-edge-ai-stt/enph-479-edge-ai-stt.git"
REPO_DIR = "/content/enph-479-edge-ai-stt"
BRANCH = "main"  # set to a feature branch to test it before merging

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
# Always land on the tip of BRANCH, even if an earlier run left the clone on another branch.
git = ["git", "-C", REPO_DIR]
subprocess.run([*git, "fetch", "origin", BRANCH], check=True)
subprocess.run([*git, "checkout", "-B", BRANCH, f"origin/{BRANCH}"], check=True)
print(subprocess.run([*git, "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

# Installs this checkout's package and its dependencies (jiwer, soundfile);
# torch and torchaudio come with Colab.
training_dir = os.path.join(REPO_DIR, "training")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", training_dir], check=True)

# The editable install only takes effect at interpreter start, so put src/ on the
# path for this already-running kernel.
sys.path.insert(0, os.path.join(training_dir, "src"))
importlib.invalidate_caches()

## 2. Mount Drive and configure the run

Mounting Drive here means the permission prompt comes up now, not after training. The results go to `MyDrive/enph479-stt/runs/<start time>-cm/` at the end.

`KEEP` is the fraction of the corpus's 40.4M sentences (4.25B characters) to train on. 0.05 gives about 2M sentences, 210M characters, roughly the size of the WSJ LM text the paper used. Not yet timed on a T4: check ms/chunk in the first progress lines and adjust `epochs` to fit the session.

In [ ]:
import time

from google.colab import drive

from training.cm.train import TrainConfig

drive.mount("/content/drive")
RUN_DIR = f"/content/drive/MyDrive/enph479-stt/runs/{time.strftime('%Y%m%d-%H%M')}-cm"

KEEP = 0.05
cfg = TrainConfig(epochs=5)
print(RUN_DIR)
cfg

## 3. Data

Downloads the normalized LM text (1.5 GB gzip, kept gzipped) and dev-clean (337 MB, for its transcripts only) to `/content/data`. Streams the corpus once to sample `KEEP` of its sentences, then encodes both sides to EOS-joined index streams. The LM text excludes the dev and test books, so dev-clean is held out.

In [ ]:
from training.am.dataset import list_utterances
from training.cm.dataset import sample_sentences, to_stream
from training.data.librispeech import download_lm_text, download_subset

train_sents = sample_sentences(download_lm_text("/content/data"), keep=KEEP)
dev_sents = [text for _, text in list_utterances(download_subset("dev-clean", "/content/data"))]

streams = {"train": to_stream(train_sents), "test": to_stream(dev_sents)}
print(f"train {len(train_sents):,} sentences, {len(streams['train']):,} chars")
print(f"dev {len(dev_sents):,} sentences, {len(streams['test']):,} chars")
del train_sents

## 4. Train

Bits per character starts near log2(30) = 4.9 (uniform), drops quickly past the unigram level (~4.1) as the model picks up spelling, then improves slowly as it picks up words. Each epoch line also prints a sampled line of text, which should read as more and more English-like. Progress lines show ms/chunk; each epoch line shows train and eval time.

In [ ]:
from training.cm.train import train

model = train(cfg, streams, "train.log")

## 5. Save to Drive

Writes `cm.pt` (the weights) and `train.log` to the run folder, then flushes Drive so the upload finishes before the session ends.

In [ ]:
import shutil

import torch

os.makedirs(RUN_DIR, exist_ok=True)
torch.save({"model": model.state_dict()}, f"{RUN_DIR}/cm.pt")
shutil.copy("train.log", RUN_DIR)
print(f"saved cm.pt and train.log to {RUN_DIR}")
drive.flush_and_unmount()